In [2]:
import os

os.makedirs("src", exist_ok=True)
os.makedirs("data/processed_v2", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs("docs", exist_ok=True)

print("Structure created")

Structure created


In [3]:
# --- Install deps (мінімальні, без NLP-бібліотек!) ---
!pip install pandas tqdm

In [4]:
%%writefile src/preprocess.py
import re
from typing import List, Dict

URL_RE = re.compile(r'https?://\\S+')
EMAIL_RE = re.compile(r'\\S+@\\S+')
PHONE_RE = re.compile(r'\\+?\\d[\\d\\s\\-\\(\\)]{7,}')
MULTISPACE_RE = re.compile(r'\\s+')

UA_ABBREVIATIONS = {"м.", "вул.", "р.", "т.д.", "ім."}

def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\\n", " ").replace("\\t", " ")
    text = MULTISPACE_RE.sub(" ", text)
    return text.strip()

def normalize_text(text: str) -> str:
    text = text.replace("’", "'").replace("`", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("«", '"').replace("»", '"')
    return text

def mask_pii(text: str) -> str:
    text = URL_RE.sub("<URL>", text)
    text = EMAIL_RE.sub("<EMAIL>", text)
    text = PHONE_RE.sub("<PHONE>", text)
    return text

SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\\s+(?=[А-ЯA-Z])')

def sentence_split(text: str) -> List[str]:
    parts = SENT_SPLIT_RE.split(text)
    merged = []
    for part in parts:
        if merged and any(merged[-1].endswith(abbr) for abbr in UA_ABBREVIATIONS):
            merged[-1] += " " + part
        else:
            merged.append(part)
    return merged

def preprocess(text: str) -> Dict:
    clean = clean_text(text)
    norm = normalize_text(clean)
    masked = mask_pii(norm)
    sentences = sentence_split(masked)

    return {
        "clean_text": masked,
        "sentences": sentences
    }

Overwriting src/preprocess.py


In [5]:
import pandas as pd
from src.preprocess import preprocess
from tqdm import tqdm

df = pd.read_csv("data/raw.csv")

results = df["raw_text"].fillna("").apply(preprocess)

df["processed_v2"] = results.apply(lambda x: x["clean_text"])
df["sentences"] = results.apply(lambda x: " | ".join(x["sentences"]))

df.to_csv("data/processed_v2/processed_v2.csv", index=False)

print("processed_v2 generated")

processed_v2 generated


In [6]:
before_len = df["raw_text"].str.len()
after_len = df["processed_v2"].str.len()

duplicates_before = len(df) - df["raw_text"].nunique()
duplicates_after = len(df) - df["processed_v2"].nunique()

summary = f"""
LAB2 AUDIT SUMMARY

Mean length BEFORE: {before_len.mean():.2f}
Mean length AFTER: {after_len.mean():.2f}

Duplicates BEFORE: {duplicates_before}
Duplicates AFTER: {duplicates_after}

Empty AFTER: {(df['processed_v2'].str.len()==0).sum()}
"""

print(summary)

with open("docs/audit_summary_lab2.md", "w") as f:
    f.write(summary)


LAB2 AUDIT SUMMARY

Mean length BEFORE: 304.75
Mean length AFTER: 304.75

Duplicates BEFORE: 20
Duplicates AFTER: 20

Empty AFTER: 0



In [7]:
sample = df.sample(15, random_state=42)

for i, row in sample.iterrows():
    print("RAW:", row["raw_text"][:300])
    print("PROCESSED:", row["processed_v2"][:300])
    print("-"*80)

RAW: продукти харчування риба хек с/м риба хек с/м
PROCESSED: продукти харчування риба хек с/м риба хек с/м
--------------------------------------------------------------------------------
RAW: Галерейні системи Галерейні системи підвісів картин з освітленням Галерейні системи підвісів картин з освітленням, на профілях з тросовою системою, дві стіни висотою 2,75 м і довжиною 2.80 м на кінці радіус заокруглення R = 3 м, картини будуть розміщені в 3 яруси
PROCESSED: Галерейні системи Галерейні системи підвісів картин з освітленням Галерейні системи підвісів картин з освітленням, на профілях з тросовою системою, дві стіни висотою 2,75 м і довжиною 2.80 м на кінці радіус заокруглення R = 3 м, картини будуть розміщені в 3 яруси
--------------------------------------------------------------------------------
RAW: Відновлення та заправлення картриджів Canon 728, HP Q2612A Відновлення та заправлення картриджів Canon 728, HP Q2612A Учасник повинен надати в електронному (сканованому у форматі pd

In [8]:
from src.preprocess import preprocess

def test_idempotence(text):
    once = preprocess(text)["clean_text"]
    twice = preprocess(once)["clean_text"]
    return once == twice

checks = df["raw_text"].sample(50, random_state=1).apply(test_idempotence)

print("Idempotence passed:", checks.all())

Idempotence passed: True


In [9]:
import pandas as pd
import os

os.makedirs("data/sample", exist_ok=True)

df_full = pd.read_csv("data/processed_v2/processed_v2.csv")

df_sample = df_full.sample(30, random_state=42)
df_sample.to_csv("data/sample/sample_processed_v2.csv", index=False)

print("Sample created")

Sample created
